[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/10_gqa.ipynb)

# 🔴 Hard: Grouped Query Attention (GQA)

Реализуйте **Grouped Query Attention (GQA)** — компромисс между обычным Multi-Head Attention и Multi-Query Attention, уменьшающий объём K/V-проекций и KV cache при сохранении нескольких разных K/V-групп.

В обычном MHA каждой Q-голове соответствует своя K/V-голова. В GQA Q-голов больше: каждая группа из `num_heads // num_kv_heads` запросных голов совместно использует одну K- и одну V-голову. Частные случаи: `num_kv_heads == num_heads` — MHA, `num_kv_heads == 1` — Multi-Query Attention.

## Формы и экономия

Пусть `d_model=512`, `num_heads=8`, `num_kv_heads=2`. Тогда `d_k=64`, Q после reshape имеет `(B,8,S,64)`, а K/V — `(B,2,S,64)`. Каждая K/V-голова обслуживает 4 Q-головы; после `repeat_interleave` K/V получают совместимую форму `(B,8,S,64)`. Хранить cache можно до расширения, поэтому K/V-память в этом примере в 4 раза меньше, чем у MHA.

Условия корректной конфигурации: `d_model % num_heads == 0` и `num_heads % num_kv_heads == 0`. Повторение K/V не создаёт новые обучаемые представления — оно только согласует оси для матричного произведения.

### Signature
```python
class GroupQueryAttention:
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int): ...
    def forward(self, x) -> torch.Tensor:  # self-attention
```

### Requirements
- `self.W_q`: `nn.Linear(d_model, d_model)` — full Q projection
- `self.W_k`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced K projection
- `self.W_v`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced V projection
- `self.W_o`: `nn.Linear(d_model, d_model)` — output projection
- `d_k = d_model // num_heads`
- Expand KV heads with `repeat_interleave` to match Q heads
- When `num_kv_heads == num_heads`, should behave like standard MHA

## План реализации

1. Создайте Q-проекцию полного размера и уменьшенные K/V-проекции.
2. Разложите Q и K/V по их разному числу голов.
3. Повторите каждую K/V-голову ровно `num_heads // num_kv_heads` раз.
4. Выполните scaled dot-product attention, объедините Q-головы и примените `W_o`.

## Быстрые проверки

- При `(H_q,H_kv)=(8,2)` expanded K/V имеют 8 голов, но каждые четыре соседние копии равны.
- При `H_q==H_kv` слой сводится к стандартному MHA.
- Output сохраняет форму `(B,S,d_model)`.
- Некорректная делимость должна обнаруживаться сразу, а не при reshape.

## Частые ошибки

- Вычислять размер K/V-головы как `d_model // num_kv_heads`; здесь он равен `d_model // num_heads`.
- Повторять элементы внутри `d_k`, а не ось голов.
- Перемешивать порядок групп при `repeat_interleave`.
- Считать expanded K/V реальной экономией памяти cache: хранить нужно компактные головы до повторения.

## Материалы для глубокого изучения

- [GQA: Training Generalized Multi-Query Transformer Models](https://arxiv.org/abs/2305.13245) — исходная статья и сравнение MHA/MQA/GQA.
- [PyTorch scaled_dot_product_attention](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html) — параметр `enable_gqa` и ограничения backend.
- [PyTorch Transformer building blocks](https://docs.pytorch.org/tutorials/intermediate/transformer_building_blocks.html) — GQA в практической реализации decoder layer.

## Вопросы для самопроверки

1. Во сколько раз уменьшается KV cache при `H_q=16`, `H_kv=4`?
2. Почему размер одной головы определяется числом Q-голов?
3. Чем GQA отличается от MQA?

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import math

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

class GroupQueryAttention:
    def __init__(self, d_model, num_heads, num_kv_heads):
        pass  # Initialize projections

    def forward(self, x):
        pass  # Self-attention with grouped KV

In [ ]:
# 🧪 Debug
torch.manual_seed(0)
gqa = GroupQueryAttention(d_model=32, num_heads=8, num_kv_heads=2)
print("W_q shape:", gqa.W_q.weight.shape)  # (32, 32)
print("W_k shape:", gqa.W_k.weight.shape)  # (8, 32)  — only 2 KV heads * d_k=4

x = torch.randn(2, 6, 32)
out = gqa.forward(x)
print("Output shape:", out.shape)           # (2, 6, 32)

In [ ]:
from torch_judge import check
check('gqa')